# 1. Definição do problema

Este trabalho tem como objetivo desenvolver um modelo de Machine Learning para apoio ao diagnóstico médico, conforme proposto no Tech Challenge da Fase 1. A solução busca analisar dados de exames e identificar padrões que auxiliem na classificação de possíveis condições clínicas, contribuindo para a triagem inicial e suporte à decisão médica.

Para isso, será utilizado o dataset Breast Cancer Wisconsin, sugerido no enunciado, contendo características extraídas de exames relacionados ao diagnóstico de câncer de mama. O objetivo é classificar os casos como benignos ou malignos a partir das variáveis disponíveis, aplicando técnicas de análise exploratória, pré-processamento, modelagem e avaliação dos resultados.

## 2. Carregamento dos dados em um dataframe pandas

In [ ]:
import pandas as pd
df = pd.read_csv("../../datasets/dataset-wisconsin-breast-cancer.csv")
# dados.groupby("diagnosis").describe()

## 3. Inspeção inicial dos dados
- Ver shape, tipos das colunas, amostras dos dados.
- Checar valores ausentes, duplicados, outliers e distribuição das variáveis.
- Confirmar se a variável alvo está correta.


In [ ]:
df.shape # (569 linhas, 33 colunas)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.head()

A partir dessa análise inicial, verifica-se que o dataset apresenta boa consistência. Foi identificada apenas uma coluna adicional, “Unnamed”, composta exclusivamente por valores nulos, provavelmente gerados durante a exportação do arquivo CSV. Por não possuir relevância analítica, essa coluna será removida. Além disso, a coluna id também será excluída, pois não contribui para a modelagem ou para a análise dos dados.

## 4. Limpeza dos dados

In [ ]:
df = df.drop(columns=['Unnamed: 32',"id"])

Após esse procedimento, temos apenas valores não nulos conforme pode ser visto abaixo:

In [ ]:
df.isnull().sum().sort_values(ascending=False)

Verificando se existem registros duplicados no dataframe:

In [ ]:
df.duplicated().sum()

Não existem valores duplicados. Agora vamos analisar a distribuição da variável alvo `diagnosis`.

In [ ]:
df['diagnosis'].value_counts()

Existe um leve desbalanceamento neste dataset (mais benignos que malignos). Verificar se isso não ira impactar no treinamento do modelo. Além disso, será realizada a conversao da variável target `diagnosis` para formato numérico. No CSV original, essa coluna conta com valores `M` para Maligno e `B` Benigno, e portanto, será convertida para 1 e 0, respectivamente.

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
df.head()

Por fim, verificou-se que algumas colunas apresentam espaço no nome, assim com o intuito de padronizá-las, o código abaixo foi escrito.

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

# 5. Analisando correlação das variáveis

Existe um desafio nesse dataset, que é a quantidade de colunas. Isso pode dificultar a análise. Talvez seja necessário reduzir o escopo ou filtrar o que é realmente relevante. Como ponto de partida, não será verificada a correlação entre todas as variáveis entre si, mas apenas destas com a variável target `diagnosis`.

In [ ]:
corr = df.corr()[['diagnosis']].sort_values(by='diagnosis', ascending=False)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,10))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlação com a variável alvo')
plt.show()

A análise de correlação com a variável alvo indica que atributos relacionados ao tamanho e à irregularidade do tumor apresentam maior associação com o diagnóstico. Em especial, variáveis como concave points, perimeter, radius e area, principalmente em suas versões `worst` e `mean`, apresentam correlações elevadas, sugerindo forte capacidade preditiva na distinção entre casos benignos e malignos. Esses resultados são coerentes do ponto de vista clínico, uma vez que tumores malignos tendem a apresentar maior tamanho e contornos mais irregulares, o que justifica a relevância dessas características no modelo.